# Financial Crime Intelligence & Risk Analytics


In [ ]:
import pandas as pd
c=pd.read_csv('../data/processed/customer_financial_crime_intelligence.csv')
a=pd.read_csv('../data/processed/financial_crime_alerts_enriched.csv')
c.head()


## Multi-risk customers

In [ ]:
c[c['unique_risk_categories']>=2].sort_values('financial_crime_risk_score',ascending=False).head(20)


## Repeat-alert customers

In [ ]:
c[c['total_alerts']>=3].sort_values('total_alerts',ascending=False).head(20)


## Risk-category performance

In [ ]:
a.groupby('risk_category').agg(alerts=('alert_id','count'), escalation_rate=('escalated_flag','mean'))


## EDA + Feature Engineering

This section adds a simple exploratory data analysis and feature-engineering workflow to the existing **Financial Crime Intelligence & Risk Analytics** project.

The existing multi-risk customer, repeat-alert customer, and risk-category performance analysis above remains unchanged.


### 1. Dataset Review

In [ ]:
# Review the two datasets already loaded above
print("Customer intelligence dataset:", c.shape)
display(c.head())
display(c.dtypes.to_frame("data_type"))

print("\nAlert dataset:", a.shape)
display(a.head())
display(a.dtypes.to_frame("data_type"))


### 2. Missing Values, Duplicates & Data Quality

In [ ]:
# Simple data-quality review
print("Customer duplicate rows:", c.duplicated().sum())
print("Alert duplicate rows:", a.duplicated().sum())

customer_missing = c.isna().sum()
alert_missing = a.isna().sum()

print("\nCustomer fields with missing values")
display(customer_missing[customer_missing > 0].to_frame("missing_count"))

print("Alert fields with missing values")
display(alert_missing[alert_missing > 0].to_frame("missing_count"))


### 3. Summary Statistics & Range Validation

In [ ]:
# Review important numeric fields used in this project
customer_numeric = [
    col for col in
    ['unique_risk_categories', 'financial_crime_risk_score', 'total_alerts']
    if col in c.columns
]

if customer_numeric:
    display(c[customer_numeric].describe().T)

# Basic logical checks
if 'unique_risk_categories' in c.columns:
    print("Negative risk-category counts:",
          (c['unique_risk_categories'] < 0).sum())

if 'total_alerts' in c.columns:
    print("Negative alert counts:",
          (c['total_alerts'] < 0).sum())

if 'financial_crime_risk_score' in c.columns:
    print("Missing financial-crime risk scores:",
          c['financial_crime_risk_score'].isna().sum())


### 4. Simple Outlier Review

In [ ]:
# IQR review for unusual numeric values.
# Financial-crime outliers are flagged for review, not automatically removed.
outlier_summary = []

for col in ['financial_crime_risk_score', 'total_alerts', 'unique_risk_categories']:
    if col in c.columns and pd.api.types.is_numeric_dtype(c[col]):
        q1 = c[col].quantile(0.25)
        q3 = c[col].quantile(0.75)
        iqr = q3 - q1

        if iqr > 0:
            lower = q1 - 1.5 * iqr
            upper = q3 + 1.5 * iqr
            count = ((c[col] < lower) | (c[col] > upper)).sum()

            outlier_summary.append({
                'field': col,
                'potential_outliers': int(count)
            })

display(pd.DataFrame(outlier_summary))


### 5. Feature Engineering

These features are intentionally simple and directly connected to the existing financial-crime intelligence analysis.


In [ ]:
# Use copies so the original datasets and analysis stay unchanged
c_fe = c.copy()
a_fe = a.copy()

created_features = []

# Customer has more than one financial-crime risk category
if 'unique_risk_categories' in c_fe.columns:
    c_fe['multi_risk_flag'] = (c_fe['unique_risk_categories'] >= 2).astype(int)
    created_features.append('multi_risk_flag')

# Customer has repeated alerts
if 'total_alerts' in c_fe.columns:
    c_fe['repeat_alert_flag'] = (c_fe['total_alerts'] >= 3).astype(int)
    created_features.append('repeat_alert_flag')

# Simple risk band from the existing financial-crime risk score
if 'financial_crime_risk_score' in c_fe.columns:
    c_fe['risk_score_band'] = pd.cut(
        c_fe['financial_crime_risk_score'],
        bins=[-float('inf'),
              c_fe['financial_crime_risk_score'].quantile(0.50),
              c_fe['financial_crime_risk_score'].quantile(0.75),
              float('inf')],
        labels=['Lower', 'Medium', 'Higher'],
        include_lowest=True
    )
    created_features.append('risk_score_band')

print("Features created:", created_features)
display(c_fe.head())


### 6. Business Rule & KPI Validation

In [ ]:
# Validate the main business indicators already used in the project
if 'multi_risk_flag' in c_fe.columns:
    print("Multi-risk customers:", int(c_fe['multi_risk_flag'].sum()))

if 'repeat_alert_flag' in c_fe.columns:
    print("Repeat-alert customers:", int(c_fe['repeat_alert_flag'].sum()))

if {'risk_category', 'escalated_flag'}.issubset(a_fe.columns):
    risk_category_check = a_fe.groupby('risk_category').agg(
        alerts=('alert_id', 'count'),
        escalation_rate=('escalated_flag', 'mean')
    )
    display(risk_category_check)


### 7. Final Validation & Optional Export

In [ ]:
print("Final customer dataset shape:", c_fe.shape)
print("Final alert dataset shape:", a_fe.shape)
print("Customer duplicates:", c_fe.duplicated().sum())
print("Alert duplicates:", a_fe.duplicated().sum())

# Optional exports for Tableau, Streamlit, or additional analysis.
# c_fe.to_csv('../data/processed/customer_financial_crime_intelligence_enriched.csv',
#             index=False)
# a_fe.to_csv('../data/processed/financial_crime_alerts_analysis_ready.csv',
#             index=False)

print("EDA + Feature Engineering completed.")
